In [63]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns

Se tienen los dataframes para distintos porcentajes de etiquetas en los datos (5%, 10%, 20%)

In [9]:
df5train = pd.read_csv('train_5_porc.csv')
df5train

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,1.190931,-0.633681,-0.497692,0.382045,0,0,1,0,0,0
1,-1.156464,3.249031,2.098923,0.478736,0,1,1,0,1,0
2,-0.080013,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0
3,-0.014386,-0.633681,-0.497692,-0.353367,1,0,0,0,1,1
4,0.579884,1.038512,-0.497692,1.548433,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...
329,0.048854,-0.633681,-0.497692,0.986020,0,0,1,0,1,-1
330,-0.370504,-0.633681,1.140590,0.078301,1,0,0,0,1,-1
331,0.753919,-0.633681,-0.497692,-0.864595,0,1,1,0,1,-1
332,-0.014386,1.038512,1.140590,0.125453,0,1,0,0,1,-1


In [4]:
df10train = pd.read_csv('train_10_porc.csv')
df10train.head()

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,-0.148214,-0.633681,-0.497692,-0.353367,1,0,1,0,1,0
1,0.048854,-0.633681,-0.497692,-0.876361,0,1,0,1,0,1
2,1.190931,-0.633681,-0.497692,0.344365,0,0,0,0,0,1
3,0.048854,-0.633681,-0.497692,-0.939901,0,1,1,0,0,0
4,-0.080013,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0


In [5]:
df20train = pd.read_csv('train_20_porc.csv')
df20train.head()

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,-0.219200,-0.633681,-0.497692,-0.870955,0,1,1,0,1,0
1,-0.719452,-0.633681,-0.497692,0.888262,0,0,1,0,1,0
2,-0.625434,-0.633681,-0.497692,-0.593383,1,0,1,0,1,0
3,-0.014386,-0.633681,-0.497692,-0.353367,1,0,0,0,1,1
4,0.048854,1.038512,4.944562,1.284875,0,1,0,0,1,1


In [10]:
#Dataframe de prueba
dftest = pd.read_csv('test.csv')
dftest

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,0.334561,-0.633681,-0.497692,-0.389701,1,0,1,0,1,0
1,0.048854,1.038512,-0.497692,-1.044639,0,1,1,0,0,0
2,0.485866,-0.633681,-0.497692,-0.438927,1,0,1,1,0,0
3,-1.156464,-0.633681,2.098923,0.042386,0,1,1,0,1,0
4,1.513330,1.038512,-0.497692,1.365324,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
79,0.048854,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0
80,-0.451398,-0.633681,-0.497692,-0.327799,1,0,1,0,0,0
81,0.109874,-0.633681,-0.497692,0.290353,1,0,1,0,1,0
82,0.986118,-0.633681,-0.497692,-0.593383,1,0,1,0,1,0


Separación de datos:

Se sigue el proceso hecho en el archivo de EDA

In [23]:
df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [24]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

In [25]:
#Tratamiento de nulos
mediana_age = df['Age'].median()
df['Age'] = df['Age'].fillna(mediana_age)
mediana_fare = df['Fare'].median()
df['Fare'] = df['Fare'].fillna(mediana_fare)

In [26]:
#Se aplica una escala logarítmica
columnas = ['Age', 'SibSp', 'Parch', 'Fare']
for columna in columnas:
    df[columna] = np.log1p(df[columna])

In [27]:
scaler = StandardScaler()
df[columnas] = scaler.fit_transform(df[columnas])
columnas_categoricas = ['Pclass', 'Sex', 'Embarked']

df = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True, dtype=int)

In [28]:
df.head()

,Survived,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S
0,0,0.461545,-0.633681,-0.497692,-0.867031,0,1,1,1,0
1,1,0.986118,1.038512,-0.497692,-0.969149,0,1,0,0,1
2,0,1.458985,-0.633681,-0.497692,-0.669252,1,0,1,1,0
3,0,0.048854,-0.633681,-0.497692,-0.773647,0,1,1,0,1
4,1,-0.293207,1.038512,1.140590,-0.443786,0,1,0,0,1


In [29]:
target = 'Survived'
X = df.drop(columns=[target])
y = df[target]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

TRANSDUCTIVE SVM 

In [38]:
C_values = [0.1, 1, 10]
gamma_values = [0.01, 0.1, 1]
threshold_values = [0.6, 0.8, 0.95]

Para el 5%

In [39]:
y5train=df5train['Survived']
resultados1=[]
for C in C_values:
    for gamma in gamma_values:
        for threshold in threshold_values:
            
            base_svm5 = SVC(probability=True)
            
            modelo1 = SelfTrainingClassifier(
                base_svm5,
                threshold=threshold
            )
            
            modelo1.fit(X_train, y5train)
            
            y_pred5 = modelo1.predict(X_test)
            
            acc = accuracy_score(y_test, y_pred5)
            f1 = f1_score(y_test, y_pred5)
            
            resultados1.append({
                "C": C,
                "gamma": gamma,
                "threshold": threshold,
                "accuracy": acc,
                "f1": f1
            })

df_resultados5 = pd.DataFrame(resultados1)

In [40]:
df_resultados5

,C,gamma,threshold,accuracy,f1
0,0.1,0.01,0.60,0.630952,0.000000
1,0.1,0.01,0.80,0.630952,0.000000
2,0.1,0.01,0.95,0.630952,0.000000
3,0.1,0.10,0.60,0.630952,0.000000
4,0.1,0.10,0.80,0.630952,0.000000
5,0.1,0.10,0.95,0.630952,0.000000
6,0.1,1.00,0.60,0.630952,0.000000
7,0.1,1.00,0.80,0.630952,0.000000
8,0.1,1.00,0.95,0.630952,0.000000
9,1.0,0.01,0.60,0.630952,0.000000


In [100]:
df_resultados5["score"] = (
    0.7 * df_resultados5["f1"] +
    0.3 * df_resultados5["accuracy"]
)

mejor5 = df_resultados5.loc[
    df_resultados5["score"].idxmax()
]

In [101]:
print("==Mejor modelo==")
print(mejor5)

==Mejor modelo==
C            1.000000
gamma        1.000000
threshold    0.800000
accuracy     0.666667
f1           0.222222
score        0.355556
Name: 16, dtype: float64


In [102]:
cm = confusion_matrix(y_test, y_pred5)
cm

array([[53,  0],
       [31,  0]])

Se observa que este modelo, a pesar que es el mejor modelo, este es bastante deficiente en la clasificación, ya que solo ha clasificado correctamente 53 elementos, y el resto ha sido herrado (31  elementos). Cabe decir que de los que ha clasificado correctamente, todos son solamente de un tipo.

Para el 10%

In [44]:
y10train=df10train['Survived']
resultados2=[]
for C in C_values:
    for gamma in gamma_values:
        for threshold in threshold_values:
            
            base_svm10 = SVC(probability=True)
            
            modelo2 = SelfTrainingClassifier(
                base_svm10,
                threshold=threshold
            )
            
            modelo2.fit(X_train, y10train)
            
            y_pred10 = modelo2.predict(X_test)
            
            acc = accuracy_score(y_test, y_pred10)
            f1 = f1_score(y_test, y_pred10)
            
            resultados2.append({
                "C": C,
                "gamma": gamma,
                "threshold": threshold,
                "accuracy": acc,
                "f1": f1
            })

df_resultados10 = pd.DataFrame(resultados2)

In [45]:
df_resultados10

,C,gamma,threshold,accuracy,f1
0,0.1,0.01,0.60,0.630952,0.0
1,0.1,0.01,0.80,0.714286,0.4
2,0.1,0.01,0.95,0.714286,0.4
3,0.1,0.10,0.60,0.630952,0.0
4,0.1,0.10,0.80,0.630952,0.0
5,0.1,0.10,0.95,0.714286,0.4
6,0.1,1.00,0.60,0.630952,0.0
7,0.1,1.00,0.80,0.630952,0.0
8,0.1,1.00,0.95,0.714286,0.4
9,1.0,0.01,0.60,0.630952,0.0


In [103]:
df_resultados10["score"] = (
    0.7 * df_resultados10["f1"] +
    0.3 * df_resultados10["accuracy"]
)

mejor10 = df_resultados10.loc[
    df_resultados5["score"].idxmax()
]
print(mejor10)

C            1.000000
gamma        1.000000
threshold    0.800000
accuracy     0.714286
f1           0.400000
score        0.494286
Name: 16, dtype: float64


In [104]:
cm2 = confusion_matrix(y_test, y_pred10)
cm2

array([[52,  1],
       [23,  8]])

Es posible observar que este modelo no resulta ser un buen clasificador, aunque es mejor que el anterior. Esto es porque ahora ya clasifica correctamente de los dos tipos, y estos siguen siendo mayores que los casos errados. Aún así, no resulta ser un buen modelo.

Para el 20%

In [47]:
y20train=df20train['Survived']
resultados3=[]
for C in C_values:
    for gamma in gamma_values:
        for threshold in threshold_values:
            
            base_svm20 = SVC(probability=True)
            
            modelo3 = SelfTrainingClassifier(
                base_svm20,
                threshold=threshold
            )
            
            modelo3.fit(X_train, y20train)
            
            y_pred20 = modelo3.predict(X_test)
            
            acc = accuracy_score(y_test, y_pred20)
            f1 = f1_score(y_test, y_pred20)
            
            resultados3.append({
                "C": C,
                "gamma": gamma,
                "threshold": threshold,
                "accuracy": acc,
                "f1": f1
            })

df_resultados20 = pd.DataFrame(resultados3)

In [48]:
df_resultados20

,C,gamma,threshold,accuracy,f1
0,0.1,0.01,0.60,0.630952,0.000000
1,0.1,0.01,0.80,0.738095,0.450000
2,0.1,0.01,0.95,0.738095,0.450000
3,0.1,0.10,0.60,0.702381,0.358974
4,0.1,0.10,0.80,0.738095,0.450000
5,0.1,0.10,0.95,0.738095,0.450000
6,0.1,1.00,0.60,0.702381,0.358974
7,0.1,1.00,0.80,0.738095,0.450000
8,0.1,1.00,0.95,0.738095,0.450000
9,1.0,0.01,0.60,0.702381,0.358974


In [105]:
df_resultados20["score"] = (
    0.7 * df_resultados20["f1"] +
    0.3 * df_resultados20["accuracy"]
)

mejor20 = df_resultados20.loc[
    df_resultados20["score"].idxmax()
]
print(mejor20)

C            0.100000
gamma        0.010000
threshold    0.800000
accuracy     0.738095
f1           0.450000
score        0.536429
Name: 1, dtype: float64


In [106]:

cm3 = confusion_matrix(y_test, y_pred20)
cm3

array([[53,  0],
       [22,  9]])

Se observa que este tampoco es un buen modelo de clasificación.